In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    desc,
    from_utc_timestamp,
    lit,
    row_number,
    sha2,
    to_date,
    when,
)
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_underlaw_cepalbiplatform(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema=(
        #      "UnderLawId string,"
        #      "SourceSystem string,"
        #      "PortfolioKey string,"
        #      "PortfolioName string,"
        #      "IngestionTimestamp timestamp,"
        #      "BatchId string,"
        #      "UnderLawKey string,"
        #      "CDI string,"
        #      "TaxRegistrationNumber string,"
        #      "ApplicationNumber string,"
        #      "ApplicationStatusCode string,"
        #      "ApplicationStatus string,"
        #      "ApplicationStatusDate timestamp,"
        #      "TypeOfCourt string,"
        #      "CourtDate date,"
        #      "RecordDate date,"
        #      "SubmissionDate date,"
        #      "InitiationDate date,"
        #      "CourtOutcomeCode string,"
        #      "CourtOutcome string,"
        #      "CompletionReasonCode int,"
        #      "CompletionReason string,"
        #      "LawSystem string,"
        #      "PlatformName string,"
        #      "InLawFlag int,"
        #      "LawStatusCode string,"
        #      "LawStatus string,"
        #      "CalcLawStatusCode string,"
        #      "CalcLawStatus string,"
        #      "UnderLawType string,"
        #      "UnderLawTypeCode string,"
        #      "TemporaryCourtDecisionDate timestamp,"
        #      "DeductionDate timestamp,"
        #      "CountyCourtCode string,"
        #      "CountyCourt string,"
        #      "InterbankStatus3869Code string,"
        #      "InterbankStatus3869 string,"
        #      "SubmissionDateGroup string,"
        #      "ResubmissionStatusFlag string,"
        #      "ResubmissionStatusCode string, "
        #      "ResubmissionStage string, "
        #      "AppealCourtDate timestamp, "
        #      "CepalEligibilityDate timestamp, "
        #      "BusinessDate date"
        #    )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.UnderLaw"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "cepalbiplatform",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== cepalbiplatform_l4738allapplication (driver, guide §2) ==================

        allapplication_extra_filters = None

        if allapplication_extra_filters is not None:
            allapplication_extra_filters.append(date_filter)
            allapplication_final_filters = " AND ".join(allapplication_extra_filters)
        else:
            allapplication_final_filters = date_filter

        # BusinessDate is applied via the filter, not put in selected_fields
        # (new-curated-table-checklist). We still read it for the final select.
        allapplication_selected_fields = [
            "ApplicationNumber",
            "LawSystemId",
            "Status",
            "StatusStartDate",
            "InitiationDate",
            "SubmissionDate",
            "ApplicantVatNumber",
            "CompletionReasonId",
            "CompletionReason",
            "StagingKey",
            "BusinessDate",
        ]

        applications = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cepalbiplatform_l4738allapplication",
            allapplication_final_filters,
            allapplication_selected_fields,
        )

        # Category scope (guide §1): drop any LawSystemId not owned by this notebook.
        applications = applications.filter(
            "LawSystemId IN (1, 2, 3, 4, 5, 6, 7, 9, 10, 20)"
        )

        # ================== Historicity (guide §4.2) ==================
        # Prefer a progressed row over a Status == 1 draft, otherwise take the row
        # with Status != 1 and (Status < 98 OR Status == 99). Then dedupe to one
        # row per ApplicationNumber with the prescribed window.
        applications_progressed = applications.filter(
            "((Status > 1 AND Status < 98) OR "
            "(Status == 99))"
        ).distinct()

        historicity_window = Window.partitionBy("ApplicationNumber").orderBy(
            desc("StatusStartDate"), desc("StagingKey")
        )

        applications_final_df = (
            applications_progressed.withColumn(
                "row_num", row_number().over(historicity_window)
            )
            .filter("row_num == 1")
            .drop("row_num")
        )

        # ================== Category mapping (guide §1, keyed on LawSystemId) ==================
        applications_final_df = (
            applications_final_df.withColumn(
                "LawSystem",
                when(applications_final_df.LawSystemId == 1, lit("Εξωδικαστικός μηχανισμός"))
                .when(applications_final_df.LawSystemId == 2, lit("Επιδότηση δόσης κύριας κατοικίας"))
                .when(applications_final_df.LawSystemId == 3, lit("Εξυγίανση"))
                .when(applications_final_df.LawSystemId == 4, lit("Πτώχευση"))
                .when(applications_final_df.LawSystemId == 5, lit("Βεβαίωση ευάλωτου"))
                .when(applications_final_df.LawSystemId == 6, lit("Μηχανισμός έγκαιρης προειδοποίησης"))
                .when(applications_final_df.LawSystemId == 7, lit("Δημοσιεύσεις"))
                .when(applications_final_df.LawSystemId == 9, lit("Βεβαίωση οφειλών"))
                .when(applications_final_df.LawSystemId == 10, lit("Κώδικας Δεοντολογίας"))
                .when(applications_final_df.LawSystemId == 20, lit("Επιδότηση δόσης λόγω αύξησης επιτοκίων"))
                .otherwise(lit(None)),
            )
            .withColumn(
                "UnderLawType",
                when(applications_final_df.LawSystemId == 10, lit("N4224/2013 (CoC)"))
                .when(applications_final_df.LawSystemId == 20, lit("Bridge3"))
                .otherwise(lit("N4738/2020")),
            )
            .withColumn(
                "PlatformName",
                when(applications_final_df.LawSystemId == 10, lit("CoC"))
                .when(applications_final_df.LawSystemId == 20, lit("Bridge3"))
                .otherwise(lit("L4738")),
            )
        )

        # ================== Status literal (guide Appendix A) ==================
        # Status 3 has two labels; Πτώχευση (LawSystemId == 4) uses a different label.
        applications_final_df = applications_final_df.withColumn(
            "StatusLiteral",
            when(applications_final_df.Status == 1, lit("1 - Δημιουργία και συμπλήρωση αίτησης"))
            .when(applications_final_df.Status == 2, lit("2 - Αξιολόγηση αίτησης και υποβολή πρότασης αναδιάρθρωσης από χρηματοδοτικούς φορείς"))
            .when((applications_final_df.Status == 3) & (applications_final_df.LawSystemId == 4), lit("3 - Παρακολούθηση αίτησης"))
            .when(applications_final_df.Status == 3, lit("3 - Ανάρτηση συμβατικών εγγράφων"))
            .when(applications_final_df.Status == 4, lit("4 - Αξιολόγηση πρότασης αναδιάρθρωσης από οφειλέτη"))
            .when(applications_final_df.Status == 5, lit("5 - Αξιολόγηση αίτησης και πρότασης αναδιάρθρωσης από φορείς Δημοσίου"))
            .when(applications_final_df.Status == 6, lit("6 - Διμερής διαδικασία - Αξιολόγηση αίτησης και πρότασης αναδιάρθρωσης από φορείς Δημοσίου"))
            .when(applications_final_df.Status == 7, lit("7 - Διμερής διαδικασία - Αξιολόγηση πρότασης αναδιάρθρωσης από οφειλέτη"))
            .when(applications_final_df.Status == 8, lit("8 - Προς Αναμονή Διμερούς Διαπραγμάτευσης"))
            .when(applications_final_df.Status == 9, lit("9 - Επεξεργασία Αίτησης από το Σύστημα"))
            .when(applications_final_df.Status == 10, lit("10 - Διαδικασία Διαμεσολάβησης"))
            .when(applications_final_df.Status == 98, lit("98 - Διμερής διαδικασία – Επιτυχής διαπραγμάτευση"))
            .when(applications_final_df.Status == 99, lit("99 - Ολοκλήρωση αίτησης"))
            .otherwise(applications_final_df.Status.cast("string")),
        )

        # ================== cepalbiplatform_l4738listofvalues (guide §4.4) ==================

        listofvalues_extra_filters = None

        if listofvalues_extra_filters is not None:
            listofvalues_extra_filters.append(date_filter)
            listofvalues_final_filters = " AND ".join(listofvalues_extra_filters)
        else:
            listofvalues_final_filters = date_filter

        listofvalues_selected_fields = [
            "CategoryId",
            "ValueId",
            "Value",
        ]

        listofvalues = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cepalbiplatform_l4738listofvalues",
            listofvalues_final_filters,
            listofvalues_selected_fields,
        )

        # Completion-reason literal lives under CategoryId = 25 (Λόγος ολοκλήρωσης αίτησης).
        completion_reason = listofvalues.filter("CategoryId == 25").distinct()

        # ================ join ===========

        underlaw_df = applications_final_df.join(
            completion_reason,
            applications_final_df.CompletionReasonId == completion_reason.ValueId,
            "left",
        )

        # ================  Primary Key (guide §4.8)  ================

        UnderLawId_primary_key_fields = [
            applications_final_df.ApplicationNumber,
            applications_final_df.UnderLawType,
            applications_final_df.LawSystem,
            applications_final_df.ApplicantVatNumber,
            coalesce(lit(None).cast("string"), lit("-1")),
            lit("cepalbiplatform"),
        ]

        UnderLawKey_primary_key_fields = [
            applications_final_df.ApplicationNumber,
            applications_final_df.UnderLawType,
            applications_final_df.LawSystem,
            applications_final_df.ApplicantVatNumber,
            coalesce(lit(None).cast("string"), lit("-1")),
        ]

        underlaw_df = (
            underlaw_df.withColumn(
                "UnderLawId",
                sha2(
                    concat_ws(";", *UnderLawId_primary_key_fields),
                    256,
                ),
            )
            .withColumn("UnderLawKey", concat_ws(";", *UnderLawKey_primary_key_fields))
            .withColumn(
                "SubmissionDate",
                when(
                    to_date(underlaw_df.SubmissionDate, "yyyy-MM-dd")
                    > lit("1900-01-01"),
                    to_date(underlaw_df.SubmissionDate, "yyyy-MM-dd"),
                ).otherwise(lit(None)),
            )
        )

        # ================== Selection of Final Columns (guide §4.9) ==================

        underlaw_cepalbiplatform = underlaw_df.select(
            underlaw_df.UnderLawId.alias("UnderLawId"),
            lit("cepalbiplatform").alias("SourceSystem"),
            lit(None).cast("string").alias("PortfolioKey"),
            lit(None).cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            underlaw_df.UnderLawKey.alias("UnderLawKey"),
            lit(None).cast("string").alias("CDI"),
            underlaw_df.ApplicantVatNumber.alias("TaxRegistrationNumber"),
            applications_final_df.ApplicationNumber.alias("ApplicationNumber"),
            underlaw_df.Status.cast("string").alias("ApplicationStatusCode"),
            underlaw_df.StatusLiteral.cast("string").alias("ApplicationStatus"),
            underlaw_df.StatusStartDate.cast("timestamp").alias(
                "ApplicationStatusDate"
            ),
            lit(None).cast("string").alias("TypeOfCourt"),
            lit(None).cast("date").alias("CourtDate"),
            lit(None).alias("RecordDate"),
            underlaw_df.SubmissionDate.alias("SubmissionDate"),
            underlaw_df.InitiationDate.cast("date").alias("InitiationDate"),
            lit(None).cast("string").alias("CourtOutcomeCode"),
            lit(None).cast("string").alias("CourtOutcome"),
            underlaw_df.CompletionReasonId.alias("CompletionReasonCode"),
            underlaw_df.Value.alias("CompletionReason"),
            underlaw_df.LawSystem.alias("LawSystem"),
            underlaw_df.PlatformName.alias("PlatformName"),
            lit(None).cast("int").alias("InLawFlag"),
            lit(None).cast("string").alias("LawStatusCode"),
            lit(None).cast("string").alias("LawStatus"),
            lit(None).alias("CalcLawStatusCode"),
            lit(None).alias("CalcLawStatus"),
            underlaw_df.UnderLawType.alias("UnderLawType"),
            underlaw_df.UnderLawType.alias("UnderLawTypeCode"),
            lit(None).alias("TemporaryCourtDecisionDate"),
            lit(None).alias("DeductionDate"),
            lit(None).alias("CountyCourtCode"),
            lit(None).alias("CountyCourt"),
            lit(None).alias("InterbankStatus3869Code"),
            lit(None).alias("InterbankStatus3869"),
            lit(None).alias("SubmissionDateGroup"),
            lit(None).alias("ResubmissionStatusFlag"),
            lit(None).alias("ResubmissionStatusCode"),
            lit(None).alias("ResubmissionStage"),
            lit(None).alias("AppealCourtDate"),
            lit(None).alias("CepalEligibilityDate"),
            lit(run_date).cast("date").alias("BusinessDate"),
        )
        underlaw_cepalbiplatform = underlaw_cepalbiplatform.distinct()

        # ========================== Append ==========================
        underlaw_cepalbiplatform.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

        logging.info("We did it!")


In [0]:
load_curated_underlaw_cepalbiplatform(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)